## 肝脏数据集
1. crop数据
2. normalize数据，这一部分根据需要进行

In [1]:
import math
import os
import random

import numpy as np
from torch.utils.data import Dataset, DataLoader
import nibabel
from scipy import ndimage
import time
import torch
import torch.nn as nn

In [2]:
class LiverDS(Dataset):
    def __init__(self, img_list, phase,crop_size, scale_size):
        self.series_list = []
        self.images_list = []
        self.masks_list = []
        self.phase = phase
        self.crop_size = crop_size
        self.scale_size = scale_size
        with open(img_list, 'r') as f:
            for line in f.readlines():
                line = line.strip()
                if line is None or len(line) == 0:
                    continue
                ss = line.split('\t')
                if len(ss) != 2:
                    continue
                self.images_list.append(ss[0])
                self.masks_list.append(ss[1])
                
    def __random_crop_data(self, volume, mask, size):
        [img_d, img_h, img_w] = volume.shape
        [input_d, input_h, input_w] = size
        z_min_upper = img_d - input_d
        y_min_upper = img_h - input_h
        x_min_upper = img_w - input_w
        Z_min = np.random.randint(0, z_min_upper)
        Y_min = np.random.randint(0, y_min_upper)
        X_min = np.random.randint(0, x_min_upper)

        Z_max = Z_min + input_d
        Y_max = Y_min + input_h
        X_max = X_min + input_w
#         print('x:[{}-{}]\ty:[{}-{}]\tz:[{}-{}]'.format(X_min, X_max, Y_min, Y_max, Z_min, Z_max))
#         print(mask.dtype)
        print(mask.max())
        return volume[Z_min: Z_max, Y_min: Y_max, X_min: X_max], mask[Z_min: Z_max, Y_min: Y_max, X_min: X_max]
    
    def __len__(self):
        return len(self.images_list)
    
    def __getitem__(self, idx):
        if self.phase == 'train':
            volume_path = self.images_list[idx]
            mask_path = self.masks_list[idx]
            with open(volume_path, 'rb') as f:
                    volume_data = np.load(f)
            with open(mask_path, 'rb') as f:
                    mask_data = np.load(f)
            cropped_volume, cropped_mask = self.__random_crop_data(volume_data, mask_data, self.crop_size)
            cropped_volume = torch.from_numpy(cropped_volume).float()
            return cropped_volume, cropped_mask

In [3]:
# 构建配置体数据和mask数据的配置文件
data_root = '../data/processed_liver'
series_list = []
for sub_root_name in os.listdir(data_root):
    sub_root = os.path.join(data_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    series_info = os.path.join(sub_root, 'scalex1')
    if not os.path.isdir(series_info):
        continue
    mask_path = os.path.join(series_info, 'mask/bone.npy')
    volume_path = os.path.join(series_info, 'volume/volume.npy')
    if not os.path.isfile(mask_path):
        print('{} not exist!'.format(mask_path))
    if not os.path.isfile(volume_path):
        print('{} not exist!'.format(volume_path))
    series_list.append('{}\t{}'.format(volume_path, mask_path))
config_root = '../data/processed_liver/config/scalex1'
os.makedirs(config_root, exist_ok=True)
with open(os.path.join(config_root, 'series_list.txt'), 'w') as f:
    f.write('\n'.join(series_list))

In [4]:
config_file = '../data/processed_liver/config/scalex1/series_list.txt'
crop_size = [56, 448, 448]
ds = LiverDS(config_file, 'train', crop_size, crop_size)
dataloader = DataLoader(ds, batch_size=1, num_workers=1, shuffle=True, pin_memory=True)
for index, (images, masks) in enumerate(dataloader):
    print(images.shape)

1
255
torch.Size([1, 56, 448, 448])
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
1
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
1
torch.Size([1, 56, 448, 448])
1
torch.Size([1, 56, 448, 448])
255
torch.Size([1, 56, 448, 448])
